In [ ]:
from pathlib import Path
import re
import pybedtools

BASE   = Path("/home/bulat_kharisov/inf")
DATA   = BASE / "data"
BEDDIR = BASE / "results" / "bed"
OUT    = BASE / "results" / "distribution"
OUT.mkdir(parents=True, exist_ok=True)

CHROM = "1"
PROMOTER_UP, DOWNSTREAM = 1000, 200


sizes = {l.split("\t")[0]: int(l.split("\t")[1]) for l in open(DATA / "genome.fa.fai")}
SIZES = OUT / "chrom.sizes"
SIZES.write_text(f"{CHROM}\t{sizes[CHROM]}\n")
print(f"chr{CHROM}: {sizes[CHROM]:,} bp")

In [ ]:
gene_rows, exon_rows = [], []
for line in open(DATA / "genes.gff3"):
    if line.startswith("#"):
        continue
    c = line.rstrip("\n").split("\t")
    if len(c) < 9 or c[0] != CHROM:
        continue
    s, e, strand = int(c[3]) - 1, int(c[4]), c[6]
    if c[2] == "gene":
        gene_rows.append(f"{c[0]}\t{s}\t{e}\t.\t.\t{strand}")
    elif c[2] in ("exon", "CDS"):
        exon_rows.append(f"{c[0]}\t{s}\t{e}\t.\t.\t{strand}")

g = str(SIZES)
genes      = pybedtools.BedTool("\n".join(gene_rows), from_string=True).sort()
exons      = pybedtools.BedTool("\n".join(exon_rows), from_string=True).sort().merge()
introns    = genes.subtract(exons)
promoters  = genes.flank(g=g, l=PROMOTER_UP, r=0, s=True)
downstream = genes.flank(g=g, l=0, r=DOWNSTREAM, s=True)
genic      = genes.cat(promoters, downstream, postmerge=True)
intergenic = genic.complement(g=g)

REGIONS = {"Exons": exons, "Introns": introns, "Promoters": promoters,
           "Downstream": downstream, "Intergenic": intergenic}
for k, v in REGIONS.items():
    print(f"{k:11s}: {len(v)} интервалов")

In [ ]:
def load(path, merge=False):
    bt = pybedtools.BedTool(str(path)).filter(lambda x: x.chrom == CHROM).saveas().sort()
    return bt.merge() if merge else bt

STRUCT = {
    "G4":       load(BEDDIR / "g4.bed"),
    "Zhunt":    load(BEDDIR / "zhunt.bed", merge=True),
    "ZDNABERT": load(BEDDIR / "zdnabert.bed", merge=True),
}
for k, v in STRUCT.items():
    print(f"{k}: {len(v)}")

In [ ]:
import csv

TOTAL = {s: len(sbt) for s, sbt in STRUCT.items()}


header = f"{'':11s}" + "".join(f"{s+' (n)':>12s}{s+' (доля)':>14s}" for s in STRUCT)
print(header)
table = []
for rname, rbt in REGIONS.items():
    row = {}
    for s, sbt in STRUCT.items():
        n = len(sbt.intersect(rbt, u=True))
        frac = n / TOTAL[s] if TOTAL[s] else 0.0
        row[s] = (n, frac)
    table.append((rname, row))
    print(f"{rname:11s}" + "".join(f"{row[s][0]:>12d}{row[s][1]:>14.4f}" for s in STRUCT))

with open(OUT / "distribution_chr1.csv", "w", newline="") as f:
    w = csv.writer(f)
    head = ["region"]
    for s in STRUCT:
        head += [f"{s}_n", f"{s}_frac"]
    w.writerow(head)
    for rname, row in table:
        line = [rname]
        for s in STRUCT:
            line += [row[s][0], round(row[s][1], 4)]
        w.writerow(line)
print("\nвсего структур:", TOTAL)
print("сохранено ->", OUT / "distribution_chr1.csv")

In [ ]:
N = 10
exp = {s: {r: 0 for r in REGIONS} for s in STRUCT}
for s, sbt in STRUCT.items():
    for i in range(N):
        sh = sbt.shuffle(g=str(SIZES), chrom=True, seed=i + 1)
        for r, rbt in REGIONS.items():
            exp[s][r] += len(sh.intersect(rbt, u=True))

print("obs/exp (>1 обогащение, <1 обеднение)")
print(f"{'':11s}" + "".join(f"{s:>10s}" for s in STRUCT))
bg = []
for r, rbt in REGIONS.items():
    vals = {}
    for s, sbt in STRUCT.items():
        obs = len(sbt.intersect(rbt, u=True))
        e = exp[s][r] / N
        vals[s] = obs / e if e else float("nan")
    bg.append((r, vals))
    print(f"{r:11s}" + "".join(f"{vals[s]:>10.2f}" for s in STRUCT))

with open(OUT / "background_chr1.csv", "w", newline="") as f:
    w = csv.writer(f)
    w.writerow(["region"] + [f"{s}_obs_exp" for s in STRUCT])
    for r, vals in bg:
        w.writerow([r] + [round(vals[s], 2) for s in STRUCT])
print("\nсохранено ->", OUT / "background_chr1.csv")